# 4. Model Selection and Training

This notebook develops, compares, and selects a machine learning regression model for forecasting actual production quantity (`actual_qty`) in the shoe manufacturing division of DI Leather.

The objective is to identify a model that can learn relationships between production resources, operating conditions, product characteristics, and actual output, then generalize those relationships to later production records.

The notebook is divided into two parts.

### Part A — Model Selection

1. Load the engineered feature dataset.
2. Separate the predictor variables and target variable.
3. Divide the records chronologically into training and testing sets.
4. Build a preprocessing pipeline for numerical and categorical variables.
5. Train seven candidate regression models.
6. Evaluate and compare the models using R², MAE, and RMSE.
7. Compare the candidate models using their chronological test-set performance and generalization behaviour, then select the most suitable model for further tuning.

### Part B — Final Training and Saving

1. Tune the selected Gradient Boosting model using `RandomizedSearchCV` with five-fold `TimeSeriesSplit` cross-validation.
2. Retrain the selected tuned pipeline using all available records.
3. Save the complete preprocessing and modelling pipeline to `../models/production_model.joblib`.
4. Reload the saved pipeline and verify that it generates predictions successfully.

### Target Variable and Prediction Objective

The target variable is `actual_qty`, representing actual production quantity. The selected model estimates this quantity from operational, product-related, categorical, and temporal input features.

Its predictions can subsequently be compared with planned production quantities to derive expected target-achievement ratios and production gaps for the prescriptive analytics stage.

### Model Evaluation Strategy

Model performance is assessed using a chronological holdout test set. The first 7,596 records are used for training and model selection, while the most recent 1,899 records are reserved for testing.

This approach evaluates model performance on later production observations without randomly shuffling the records. Hyperparameter tuning uses five-fold `TimeSeriesSplit` cross-validation on the training data.

The evaluation metrics are:

- **R² (Coefficient of Determination):** Measures the proportion of variation in actual production quantity explained by the model.
- **MAE (Mean Absolute Error):** Measures the average absolute difference between actual and predicted production quantities.
- **RMSE (Root Mean Squared Error):** Measures prediction error while giving greater weight to larger errors.

The final tuned pipeline is retrained using all available records and saved for use in Notebook 5. The reported test metrics are obtained from the earlier chronological holdout evaluation, while the final retraining and prediction checks demonstrate that the saved pipeline can be reused for new production inputs.

# 4.1 Part A – Model Selection

Model selection involves comparing different algorithms to determine which provides the most reliable predictions for the production forecasting problem.

No single algorithm performs best for every dataset. Some models are designed to capture straightforward linear relationships, while others can learn complex interactions and nonlinear patterns.

Seven regression models are compared using the same training and testing periods and the same preprocessing approach. This provides a consistent basis for evaluating their predictive performance.

The comparison considers both prediction error and generalisation. A model that performs extremely well on training data but substantially worse on unseen data may have overfitted the training observations.

The objective is therefore not simply to select the most complex model or the model with the highest training score. It is to identify the model that performs best on the held-out test period while maintaining a reasonable difference between training and testing performance.

### 4.1.1 Import Libraries and Required Functions

The required libraries are imported for data manipulation, preprocessing, regression modelling, performance evaluation, and model saving.

Scikit-learn provides the regression algorithms, preprocessing transformers, pipelines, and evaluation metrics. XGBoost provides an additional gradient-boosting implementation, while Joblib is used to save and reload the final model.

The evaluation functions include:

- `r2_score`: Measures how well the predictions explain variation in the target.
- `mean_absolute_error`: Calculates the average absolute prediction error.
- `mean_squared_error`: Calculates the mean squared prediction error, from which RMSE is obtained.

The `Pipeline` and `ColumnTransformer` components allow preprocessing and modelling to be managed as one workflow. This ensures that transformations are fitted on the training data rather than independently using information from the test data.

The imported functions provide a consistent framework for comparing the candidate models and preserving the selected model for future predictions.

In [ ]:
# DataFrame.to_markdown() needs the 'tabulate' package:  pip install tabulate

import warnings
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')

### 4.1.2 Load the Feature Dataset

The engineered feature dataset is loaded from `final_features.csv`, and `week_start_date` is parsed as a datetime column.

The dataset contains 9,495 records and 18 columns, including the target variable, the date used to preserve chronological order, and the prepared predictors.

An assertion checks that `week_start_date` is in ascending chronological order. If the records are not correctly ordered, the notebook stops instead of proceeding with an invalid time-based split.

This validation is important because the project forecasts production quantities over time. Training on randomly mixed records could allow observations from later periods to influence a model evaluated on earlier periods, resulting in an unrealistic assessment of forecasting performance.

The date column is retained for ordering and split verification, but it is excluded from the predictors in the model training step.

In [ ]:
# Load the final feature dataset
df = pd.read_csv(
    "../data/processed/final_features.csv",
    parse_dates=["week_start_date"]
)

# Verify chronological order
assert df["week_start_date"].is_monotonic_increasing, \
    "Data must be sorted by week_start_date"

print("Dataset shape:", df.shape)

### 4.1.3 Define the Target, Predictors, and Chronological Split

The target variable is `actual_qty`, representing the actual quantity produced for each record.

The predictors are the remaining columns after excluding `actual_qty` and `week_start_date`. The date column is excluded from the model inputs because it is used to preserve the chronological sequence rather than as a direct predictor.

The dataset is divided into two parts:

- **Training set:** The first 80% of the chronologically ordered records.
- **Testing set:** The most recent 20% of the records.

The training set contains 7,596 records, covering 18 April 2022 to 24 March 2025. The test set contains 1,899 records, covering 24 March 2025 to 30 March 2026.

A chronological split is appropriate because the intended application is forecasting future production. The model learns from earlier production records and is evaluated on a later period that it did not use for fitting.

No random shuffling is performed. This preserves the time direction of the prediction problem and gives a more realistic estimate of how the models may perform on later production observations.

The test set is reserved for model comparison and should not be used to fit preprocessing transformations or model parameters.


In [ ]:
target = 'actual_qty'

X_full = df.drop(columns=[target, 'week_start_date'])
y_full = df[target]

split_idx = int(len(df) * 0.8)
X_train, X_test = X_full.iloc[:split_idx], X_full.iloc[split_idx:]
y_train, y_test = y_full.iloc[:split_idx], y_full.iloc[split_idx:]

print(f"Train: {len(X_train):,} rows  ({df['week_start_date'].iloc[0].date()} to {df['week_start_date'].iloc[split_idx - 1].date()})")
print(f"Test : {len(X_test):,} rows  ({df['week_start_date'].iloc[split_idx].date()} to {df['week_start_date'].iloc[-1].date()})")

### 4.1.4 Why a Chronological Split Is Necessary

In a forecasting problem, the order of observations matters because future production records would not be available when predictions are made for an earlier period.

A random train–test split can mix earlier and later observations. This may allow a model to learn patterns from periods that occur after some of the records it is evaluated on, making the evaluation less representative of actual forecasting.

The chronological split avoids this problem by using earlier records for training and later records for testing.

The test results therefore indicate how well each model generalises to a later period under the current evaluation setup. They do not guarantee identical performance in every future period, particularly if production processes or operating conditions change.

The training and testing periods should remain unchanged while comparing the seven candidate models so that the results are directly comparable.


### 4.1.5 Feature Scaling and Preprocessing

Feature scaling is performed using `StandardScaler` inside the modelling pipeline. Each numerical predictor is standardized using the mean and standard deviation learned from the training data.

The standardization formula is:

$$
z = \frac{x-\mu_{\text{train}}}{\sigma_{\text{train}}}
$$

where:

- \(x\) is the original numerical feature value.
- \(\mu_{\text{train}}\) is the mean calculated from the training data.
- \(\sigma_{\text{train}}\) is the standard deviation calculated from the training data.
- \(z\) is the standardized feature value.

Standardization places numerical features on a comparable scale. It is particularly useful for Linear Regression, Ridge Regression, and Lasso Regression because differences in feature scales can affect regularization and model behaviour.

Tree-based models generally do not require feature scaling. However, the same preprocessing framework is used consistently when comparing the candidate models.

The scaler is fitted within each model pipeline using the training data, preventing information from the held-out test set from influencing the scaling parameters.

Categorical features are encoded separately using the appropriate encoding methods, while `bottleneck_flag` retains its existing binary representation.


### 4.1.6 Build the Preprocessing Pipeline

The dataset contains numerical, binary, and categorical variables. These variable types require different preprocessing methods before being supplied to the regression models.

The preprocessing operations are placed inside a `ColumnTransformer`, which is then included in a modelling pipeline.

The transformations are:

| Transformer | Variables | Purpose |
|---|---|---|
| `StandardScaler` | Numerical features | Standardises each feature using the training data's mean and standard deviation. |
| `OneHotEncoder` | `financial_month`, `factory_unit`, `product_family`, `grade`, `order_category`, `size_id` | Converts categories into binary indicator columns. |
| `TargetEncoder` | `item_name` | Represents each item using a smoothed target-based encoding. |
| `passthrough` | `bottleneck_flag` | Retains the existing 0/1 representation. |

**Why use a pipeline?**

The preprocessing transformations are fitted as part of the model pipeline. When the pipeline is trained, the transformations learn from the training data rather than from the complete dataset.

This is important for preventing information leakage. For example, the scaler should not use the test set's mean and standard deviation, and target encoding should not use the held-out test targets to determine item encodings.

One-hot encoding uses `handle_unknown='ignore'`, allowing the pipeline to handle categories that were not observed during training without failing. For target encoding, unseen items can be represented using the encoder's overall target-based fallback.

Using the same preprocessing design for all candidate models makes the comparison more consistent and ensures that the final saved model contains both the required transformations and the selected regression algorithm.

In [ ]:
categorical_ohe = ['financial_month', 'factory_unit', 'product_family', 'grade', 'order_category', 'size_id']
categorical_target = ['item_name']
binary_cols = ['bottleneck_flag']
numerical_cols = [col for col in X_train.columns if col not in categorical_ohe + categorical_target + binary_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat_ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_ohe),
        ('cat_te', TargetEncoder(target_type='continuous'), categorical_target),
        ('binary', 'passthrough', binary_cols)
    ])

print("Numeric columns:", numerical_cols)
print("Binary columns :", binary_cols)

### 4.1.7 Candidate Regression Models

Seven regression models are evaluated to compare different approaches to predicting actual production quantity.

The models cover three main families:

| Model family | Candidate models | Main purpose |
|---|---|---|
| Linear models | Linear Regression, Ridge Regression, Lasso Regression | Establish a simple relationship between predictors and the target, with regularisation options. |
| Bagged tree models | Random Forest, Extra Trees | Capture nonlinear relationships and interactions using collections of decision trees. |
| Boosted tree models | Gradient Boosting, XGBoost | Build an ensemble sequentially, improving predictions by learning from earlier errors. |

Including different model families is useful because the relationships between workforce, machine availability, operating conditions, and production quantity may not be entirely linear.

Linear models provide interpretable reference points, while tree-based models can learn more complex relationships. Comparing them helps establish whether additional model complexity improves prediction on the held-out production period.

The models are initially compared using the settings specified in the code. Hyperparameter tuning is not performed in this notebook, so the results represent a comparison of these particular model configurations rather than an exhaustive search for the best possible configuration of each algorithm.


### 4.1.8 Rationale for Selecting Each Candidate Model

Each candidate model is included for a specific reason. Comparing them helps determine which modelling approach best represents the production data.

**1. Linear Regression – Baseline Model**

Linear Regression is selected as the baseline model. It estimates a linear relationship between the predictors and `actual_qty`.

The model provides a straightforward reference point for evaluating whether more complex algorithms improve forecasting performance. It is relatively simple to interpret and can show how well the target can be predicted using additive relationships between the prepared features.

Its main limitation is that it cannot automatically capture complex nonlinear relationships unless those relationships are represented through the input features. Nevertheless, it is an important starting point because a complex model should demonstrate a meaningful improvement over a simple alternative.

**2. Ridge Regression – Regularised Linear Model**

Ridge Regression is included to assess whether regularisation improves the linear baseline.

It adds an L2 penalty to the regression objective, discouraging excessively large coefficients. This can help when predictors contain overlapping information or when the model is sensitive to the combination of input variables.

Ridge retains all predictors but can shrink their coefficients towards zero. It is therefore useful for examining whether a regularised linear relationship provides better generalisation than ordinary Linear Regression.

**3. Lasso Regression – Feature-Shrinking Linear Model**

Lasso Regression uses an L1 penalty, which can shrink some coefficients exactly to zero.

This makes it useful for assessing whether a simpler linear model can achieve competitive performance while reducing the influence of less useful predictors. It is particularly relevant when the feature set contains several related variables.

However, Lasso may select one variable from a group of correlated predictors and reduce the coefficients of others. Its performance therefore depends on the data and the regularisation strength.

**4. Random Forest – Bagged Decision Trees**

Random Forest combines predictions from many decision trees trained using randomised samples and feature selections.

It is selected because production relationships may be nonlinear. For example, the relationship between workforce size and output may change depending on machine count, working hours, or other operating conditions.

By averaging predictions from multiple trees, Random Forest can capture interactions and nonlinear patterns without requiring them to be specified manually. Its main risk is overfitting, especially when the trees become too complex relative to the available data.

**5. Gradient Boosting – Sequential Tree Ensemble**

Gradient Boosting builds a sequence of decision trees, with each new tree attempting to reduce the errors made by the current ensemble.

It is selected because it can model nonlinear relationships and interactions while progressively improving the fit to the training data. Its sequential learning process differs from the parallel tree-building approach used by bagged ensembles.

Gradient Boosting often provides a useful balance between predictive flexibility and generalisation when its learning rate, tree complexity, and number of boosting stages are appropriate.

Its performance still depends on the model settings, so the test results must be examined rather than assuming it will outperform the other candidates.

**6. Extra Trees – Highly Randomised Tree Ensemble**

Extra Trees, or Extremely Randomised Trees, combines many decision trees while introducing additional randomness into the choice of split points.

It is included to compare another tree-ensemble approach with Random Forest. The extra randomness can reduce variance and sometimes improve generalisation, although it may also make individual trees less precise.

Extra Trees can capture complex nonlinear patterns and interactions. However, a large gap between training and test performance would indicate that the model fits the training data much more closely than the unseen period.

**7. XGBoost – Optimised Gradient Boosting**

XGBoost is included as an additional gradient-boosting implementation designed for efficient and flexible tree-based learning.

Like Gradient Boosting, it builds an ensemble sequentially, improving predictions through successive boosting stages. It also provides regularisation mechanisms and other controls that can help manage model complexity.

XGBoost is a useful candidate because production data may contain interactions and nonlinear relationships that simpler linear models cannot capture.

However, its more advanced implementation does not guarantee better results. Its performance depends on its hyperparameters and the structure of the dataset, so it must be judged using the same held-out test period as the other models.

In [ ]:
# Define the seven candidate regression models
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(),
    'Lasso Regression': Lasso(max_iter=10000),
    'Random Forest': RandomForestRegressor(random_state=42, n_jobs=1),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'Extra Trees': ExtraTreesRegressor(random_state=42, n_jobs=1),
    'XGBoost': XGBRegressor(
        random_state=42,
        n_jobs=1,
        objective='reg:squarederror'
    )
}

print("Number of candidate models:", len(models))

### 4.1.9 Hyperparameter Tuning of All Candidate Models

All seven candidate regression models are tuned before selecting the final model. Each model is combined with the same preprocessing pipeline and evaluated using five-fold time-series cross-validation on the training data.

`RandomizedSearchCV` explores candidate hyperparameter configurations and selects the configuration with the lowest cross-validation Mean Absolute Error (MAE). The chronological test set is kept separate from the tuning process.

After tuning, the models are ranked by their best cross-validation MAE. Their held-out test metrics are also reported for evaluation, but the test set is not used to choose hyperparameters.

Each model explores 10 sampled configurations across five time-series folds (up to 50 cross-validation fits per model), followed by refitting the selected configuration. The total runtime depends on the dataset and computer.

### Evaluation Metrics: R², MAE, and RMSE

The tuned models are evaluated using three regression metrics:

- **R²:** Measures how much variation in actual production quantity is explained by the model; higher is generally better.
- **MAE:** Measures the average absolute prediction error in production units; lower is better.
- **RMSE:** Measures prediction error while penalising larger errors more strongly; lower is better.

Cross-validation MAE is used to rank the tuned candidates. The chronological test metrics provide a separate assessment of performance on later production records.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from xgboost import XGBRegressor
import numpy as np
import pandas as pd

# Candidate hyperparameter search spaces
param_distributions = {
    'Linear Regression': {
        'model__fit_intercept': [True, False],
        'model__positive': [True, False]
    },
    'Ridge Regression': {
        'model__alpha': [0.01, 0.1, 1.0, 10.0, 100.0],
        'model__fit_intercept': [True, False]
    },
    'Lasso Regression': {
        'model__alpha': [0.001, 0.01, 0.1, 1.0, 10.0],
        'model__fit_intercept': [True, False]
    },
    'Random Forest': {
        'model__n_estimators': [100, 200, 300],
        'model__max_depth': [None, 10, 20],
        'model__min_samples_leaf': [1, 2, 5, 10],
        'model__max_features': [1.0, 'sqrt']
    },
    'Gradient Boosting': {
        'model__n_estimators': [100, 150, 200, 250],
        'model__learning_rate': [0.03, 0.05, 0.08, 0.1],
        'model__max_depth': [2, 3, 4],
        'model__min_samples_leaf': [2, 5, 10],
        'model__subsample': [0.8, 1.0]
    },
    'Extra Trees': {
        'model__n_estimators': [100, 200, 300],
        'model__max_depth': [None, 10, 20],
        'model__min_samples_leaf': [1, 2, 5, 10],
        'model__max_features': [1.0, 'sqrt']
    },
    'XGBoost': {
        'model__n_estimators': [100, 200, 300],
        'model__learning_rate': [0.03, 0.05, 0.1],
        'model__max_depth': [2, 3, 5],
        'model__min_child_weight': [1, 3, 5],
        'model__subsample': [0.8, 1.0],
        'model__colsample_bytree': [0.8, 1.0]
    }
}

# Time-aware cross-validation; observations remain in chronological order
time_cv = TimeSeriesSplit(n_splits=5)

tuned_models = {}
tuning_results = []

for name, model in models.items():
    print(f"\\nTuning {name}...")

    pipeline = Pipeline(steps=[
        ('preprocessor', clone(preprocessor)),
        ('model', clone(model))
    ])

    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_distributions[name],
        n_iter=10,
        scoring='neg_mean_absolute_error',
        cv=time_cv,
        random_state=42,
        n_jobs=-1,
        verbose=1
    )

    search.fit(X_train, y_train)

    best_pipeline = search.best_estimator_
    tuned_models[name] = best_pipeline

    # Evaluate the tuned candidate on the chronological test set
    test_predictions = best_pipeline.predict(X_test)

    tuning_results.append({
        'Model': name,
        'Best CV MAE': -search.best_score_,
        'Test R²': r2_score(y_test, test_predictions),
        'Test MAE': mean_absolute_error(y_test, test_predictions),
        'Test RMSE': np.sqrt(mean_squared_error(y_test, test_predictions)),
        'Best Parameters': search.best_params_
    })

    print("Best CV MAE:", round(-search.best_score_, 4))
    print("Best parameters:", search.best_params_)

# Rank candidates by cross-validation MAE (lower is better)
tuning_results_df = pd.DataFrame(tuning_results).sort_values(
    'Best CV MAE', ascending=True
).reset_index(drop=True)

print("\\nTuned model comparison (ranked by CV MAE):")
display(tuning_results_df[
    ['Model', 'Best CV MAE', 'Test R²', 'Test MAE', 'Test RMSE']
].round(3))

### 4.1.10 Select the Best Tuned Model

The tuned candidates are ranked by their best cross-validation MAE. The model with the lowest cross-validation MAE is selected because it has the smallest average absolute prediction error across the time-series validation folds.

The selected model is then evaluated on the held-out chronological test set using R², MAE, and RMSE. The test set is not used to tune hyperparameters or rank candidates.

In [ ]:
# Select the model with the lowest cross-validation MAE
best_model_name = tuning_results_df.iloc[0]['Model']
best_tuned_pipeline = tuned_models[best_model_name]

# Evaluate the selected model on the chronological test set
best_test_predictions = best_tuned_pipeline.predict(X_test)

print("Selected model:", best_model_name)
print("Best CV MAE:", round(tuning_results_df.iloc[0]['Best CV MAE'], 3))
print("\\nHeld-out test performance:")
print("Test R²  :", round(r2_score(y_test, best_test_predictions), 3))
print("Test MAE :", round(mean_absolute_error(y_test, best_test_predictions), 3))
print("Test RMSE:", round(np.sqrt(mean_squared_error(y_test, best_test_predictions)), 3))

# Preserve these variable names for the diagnostic and saving cells below
tuned_pipeline = best_tuned_pipeline
tuned_test_pred = best_test_predictions

### 4.1.11 Interpreting the Tuned-Model Comparison

The comparison table reports cross-validation MAE and chronological test metrics for all seven tuned models. The model is selected using the lowest cross-validation MAE, while the test metrics describe performance on the later held-out period.

A lower MAE indicates smaller average absolute errors, a lower RMSE indicates fewer or smaller large errors, and a higher R² indicates that more variation in actual production quantity is explained. The selected model is determined from the results produced by the notebook; no algorithm is assumed to be the winner in advance.

### 4.1.12 Residual Diagnostics

Residuals are calculated as actual production quantity minus predicted production quantity on the chronological test set. The histogram and normal Q–Q plot help inspect the shape of the residual distribution. These plots are diagnostic aids and do not, by themselves, prove that all model assumptions are satisfied.

In [ ]:


import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

# Calculate residuals on the chronological test set
residuals = y_test - tuned_test_pred

# Create residual diagnostic plots
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Residual distribution
sns.histplot(residuals, kde=True, ax=axes[0])
axes[0].set_title('Distribution of Test Residuals')
axes[0].set_xlabel('Actual - Predicted Production Quantity')
axes[0].set_ylabel('Frequency')

# Normal Q-Q plot
stats.probplot(residuals, dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot of Test Residuals')

plt.tight_layout()
plt.show()

# Summary statistics
print("Residual mean:", round(residuals.mean(), 3))
print("Residual standard deviation:", round(residuals.std(), 3))


### 4.1.13 Feature Importance Analysis

Feature importance analysis is used to investigate which input variables contribute most to the predictive performance of the selected tuned model.

Permutation importance is calculated by measuring how much the model's test-set performance deteriorates when the values of one feature are randomly shuffled. A larger increase in MAE indicates that the model relies more heavily on that feature for its predictions.

This method evaluates importance at the original input-column level, so it can be used with the complete preprocessing pipeline without manually interpreting every encoded column. The analysis is performed on the held-out test set after model selection and tuning. Its purpose is interpretation, not further optimisation against the test set.

Importance values describe the model's predictive reliance on the observed data. They do not establish that a feature causes changes in production output, and correlated predictors can share or obscure importance.

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.inspection import permutation_importance

# Calculate permutation importance on the held-out test data
perm_result = permutation_importance(
    tuned_pipeline,
    X_test,
    y_test,
    scoring='neg_mean_absolute_error',
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

importance_df = pd.DataFrame({
    'Feature': X_test.columns,
    'Importance Mean': perm_result.importances_mean,
    'Importance Std': perm_result.importances_std
}).sort_values(
    'Importance Mean',
    ascending=False
).reset_index(drop=True)

display(importance_df.round(4))

# Plot the 10 most important features
top_features = importance_df.head(10).sort_values(
    'Importance Mean',
    ascending=True
)

plt.figure(figsize=(10, 6))
plt.barh(
    top_features['Feature'],
    top_features['Importance Mean'],
    xerr=top_features['Importance Std']
)
plt.xlabel('Decrease in model performance when shuffled (MAE-based)')
plt.ylabel('Feature')
plt.title(f'Permutation Feature Importance — {best_model_name}')
plt.tight_layout()
plt.show()


# 4.2 Part B – Final Training and Saving

After selecting the best tuned model in Part A, the chosen pipeline is retrained using all available production records.

During model selection, the earlier 80% of records were used for training and the most recent 20% were held out for evaluation. After model selection and evaluation are complete, the selected pipeline can be fitted on the full historical dataset.

The final pipeline contains both preprocessing transformations and the selected regression model. It is saved as `production_model.joblib` for use in Notebook 5.

The test metrics reported in Part A remain the evaluation results from the chronological holdout period. Retraining on all records does not create a new independent test result.

### 4.2.1 Retrain the Selected Model on All Records

The model with the lowest cross-validation MAE is selected automatically using the tuning results. Its complete pipeline is then fitted on all available feature records, including the historical period previously reserved for testing.

The fitted pipeline contains the preprocessing steps and selected estimator so that new prediction records are transformed consistently. The pipeline is saved as `production_model.joblib` for use in Notebook 5.

The test performance reported in Part A remains the result from the earlier chronological holdout evaluation. Training on all records does not independently verify performance on future unseen records.

In [ ]:

final_pipeline = clone(tuned_pipeline)
final_pipeline.fit(X_full, y_full)

joblib.dump(
    final_pipeline,
    "../models/production_model.joblib"
)

print(f"Final selected model ({best_model_name}) saved successfully.")


### 4.2.2 Verify the Saved Model

The saved file `production_model.joblib` is reloaded, and the loaded pipeline is used to generate predictions for the last five records in the dataset.

The resulting table compares the recorded `actual_qty` values with the model's predicted quantities. This provides a basic functional check that the saved file can be loaded and used for inference.

The final five records are part of the dataset used to train the final pipeline. Consequently, these predictions are not an independent test of predictive accuracy and should not be used to replace the test metrics reported in Part A.

This step verifies that the model file is usable and that the complete pipeline can accept the feature data and return predictions. A separate evaluation using unseen future observations would be required to measure the final model's performance after deployment.

In [ ]:

# Load the final tuned model from the models folder
loaded_model = joblib.load("../models/production_model.joblib")

# Select the last five records for a functionality check
sample = X_full.tail(5)

# Compare actual and predicted production quantities
pd.DataFrame({
    'actual_qty': y_full.tail(5).values,
    'predicted_qty': loaded_model.predict(sample).round().astype(int)
})


## 4.3 Summary

This notebook developed, tuned, compared, and saved a machine learning regression pipeline for predicting actual production quantity (`actual_qty`) in the DI Leather shoe manufacturing division.

The workflow included feature scaling, categorical encoding, chronological train–test splitting, hyperparameter tuning of seven regression models, model comparison, residual diagnostics, and feature-importance analysis.

### 4.3.1 Model Selection and Evaluation

Seven regression algorithms were considered: Linear Regression, Ridge Regression, Lasso Regression, Random Forest, Gradient Boosting, Extra Trees, and XGBoost. Linear Regression provides a baseline for comparison with the other algorithms.

Each candidate model was tuned using `RandomizedSearchCV` with five-fold `TimeSeriesSplit` cross-validation on the training data. Candidates were ranked by their best cross-validation MAE, and the selected model was evaluated on the chronological test set using R², MAE, and RMSE.

### 4.3.2 Final Model

The model with the lowest cross-validation MAE was selected based on the actual results generated by the notebook. The selected pipeline was retrained on all available records and saved as `production_model.joblib` for use in Notebook 5.

The reported test metrics remain those obtained from the chronological holdout period before final retraining. The predictions for the last five records after final training are only a functionality check, not an independent test of predictive accuracy.